In [35]:
import importlib
import json
import logging
import os
import re
import sys
import getpass
from pathlib import Path

import anthropic
import pymupdf  # fitz

import preprocessing_prompts
importlib.reload(preprocessing_prompts)
from preprocessing_prompts import (
    PROMPT_VERSION,
    SYSTEM_PROMPT,
    STUDY_METADATA_PROMPT,
    PC_OBJ_PROMPT,
    PC_ELG_CRIT_PROMPT,
    PC_EVENT_PROMPT,
    PC_ASSESSMENT_PROMPT,
    PC_EXPOSURE_PROMPT,
    PC_CONMED_PROMPT,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger(__name__)

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

print(f"Imports OK — prompt version: {PROMPT_VERSION}")

Imports OK — prompt version: v2.1


## Configuration

In [36]:
# PC types that MUST be present — raises ExtractorError if missing or empty
REQUIRED_PC_TYPES = ["PC-OBJ", "PC-ELG-CRIT", "PC-EVENT", "PC-ASSESSMENT", "PC-EXPOSURE", "PC-CONMED"]
OPTIONAL_PC_TYPES = []

# LLM settings
MODEL = "claude-opus-4-5"
MAX_TOKENS = 4096
TEMPERATURE = 0

# Max characters to send to LLM per section (stay within context limits)
MAX_SECTION_CHARS = 12_000

# Section header patterns per PC type (ordered most-specific first, matched case-insensitively)
PC_SECTION_HEADERS: dict[str, list[str]] = {
    "PC-OBJ": [
        r"\b3\.\s+objectives\s+and\s+endpoints\b",
        r"\bobjectives\s+and\s+endpoints\b",
        r"\bstudy\s+objectives\s+and\s+endpoints\b",
        r"\bstudy\s+objectives\b",
        r"\bobjectives\b",
    ],
    "PC-ELG-CRIT": [
        r"\b5\.\s+study\s+population\b",
        r"\bstudy\s+population\b",
        r"\beligibility\s+criteria\b",
        r"\binclusion\s+criteria\b",
    ],
    "PC-EVENT": [
        r"\b8\.3\s+adverse\s+events\b",
        r"\badverse\s+events\s+and\s+serious\s+adverse\s+events\b",
        r"\badverse\s+events\b",
        r"\bsafety\s+reporting\b",
    ],
    "PC-ASSESSMENT": [
        r"\b8\.\s+study\s+assessments\s+and\s+procedures\b",
        r"\bstudy\s+assessments\s+and\s+procedures\b",
        r"\bstudy\s+assessments\b",
        r"\bassessments\s+and\s+procedures\b",
    ],
    "PC-EXPOSURE": [
        r"\b6\.\s+study\s+intervention\b",
        r"\bstudy\s+intervention\b",
        r"\bdosage\s+and\s+administration\b",
        r"\bstudy\s+drug\s+administration\b",
    ],
    "PC-CONMED": [
        r"\b6\.5\s+concomitant\s+therapy\b",
        # require a standalone section-number prefix to avoid matching
        # "Prior/Concomitant Therapy" inside exclusion criteria
        r"^\s*\d+\.\d+\.?\s+concomitant\s+(?:therapy|medications)\b",
        r"\ballowed\s+medicine\s+and\s+therapy\b",
    ],
}

# Section ending patterns per PC type (stop collecting when one of these is hit)
PC_SECTION_ENDINGS: dict[str, list[str]] = {
    "PC-OBJ": [
        r"\b4\.\s+study\s+design\b",
        r"\bstudy\s+design\b",
        r"\bintroduction\b",
    ],
    "PC-ELG-CRIT": [
        r"\b6\.\s+study\s+intervention\b",
        # removed generic "\bstudy\s+intervention\b" — too broad, matches inside eligibility text
        r"\bstudy\s+drug\b",
    ],
    "PC-EVENT": [
        r"\b8\.4\b",
        r"\btreatment\s+of\s+overdose\b",
        r"\bpharmacokinetics\b",
        r"\b9\.\s+statistical\b",
    ],
    "PC-ASSESSMENT": [
        r"\b9\.\s+statistical\s+considerations\b",
        r"\bstatistical\s+considerations\b",
        r"\bsupporting\s+documentation\b",
    ],
    "PC-EXPOSURE": [
        r"\b7\.\s+discontinuation\b",
        r"\bdiscontinuation\s+of\s+study\s+intervention\b",
        r"\bstudy\s+assessments\b",
    ],
    "PC-CONMED": [
        r"\b6\.6\s+dose\s+modification\b",
        r"\bdose\s+modification\b",
        r"\b7\.\s+discontinuation\b",
    ],
}


class ExtractorError(Exception):
    """Raised when a required PC type cannot be extracted."""
    pass

## PDF Text Extraction

In [37]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """
    Extract full text from PDF using pymupdf.
    Sorts text blocks by vertical position to preserve reading order.
    Page breaks are marked as '\\n--- PAGE {n} ---\\n'.
    """
    logger.info(f"Extracting text from PDF: {pdf_path}")

    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    pages_text = []
    with pymupdf.open(str(pdf_path)) as doc:
        logger.info(f"PDF has {len(doc)} pages")
        for page_num, page in enumerate(doc, start=1):
            blocks = page.get_text("blocks", sort=True)
            page_lines = []
            for block in blocks:
                # block = (x0, y0, x1, y1, text, block_no, block_type)
                if block[6] == 0:  # type 0 = text block
                    text = block[4].strip()
                    if text:
                        page_lines.append(text)
            pages_text.append(f"\n--- PAGE {page_num} ---\n" + "\n".join(page_lines))

    full_text = "\n".join(pages_text)
    logger.info(f"Extracted {len(full_text):,} characters from PDF")
    return full_text

## Section Finding

In [38]:
def _is_toc_line(line: str) -> bool:
    """Return True if the line looks like a Table of Contents entry (dots + page number)."""
    return bool(re.search(r'\.{4,}\s*\d+\s*$', line.strip()))


# Human-readable descriptions used in the LLM fallback prompt
_PC_TYPE_DESCRIPTIONS = {
    "PC-OBJ":        "study objectives and endpoints (primary, secondary, safety, exploratory)",
    "PC-ELG-CRIT":   "eligibility criteria — inclusion criteria and exclusion criteria",
    "PC-EVENT":      "adverse events (AEs) and serious adverse events (SAEs), safety reporting",
    "PC-ASSESSMENT": "study assessments and procedures (efficacy, safety, PK, PD, biomarkers)",
    "PC-EXPOSURE":   "study intervention / drug dosing, administration, and formulation",
    "PC-CONMED":     "concomitant medications and therapy (allowed, prohibited, requiring caution)",
}


def find_section_text(full_text: str, pc_type: str) -> str | None:
    """
    Regex-based section finder. Skips ToC lines.
    Returns None if no match found (triggers LLM fallback).
    """
    lines = full_text.split("\n")
    start_patterns = PC_SECTION_HEADERS.get(pc_type, [])
    end_patterns = PC_SECTION_ENDINGS.get(pc_type, [])

    start_line_idx = None
    matched_pattern = None

    for pattern in start_patterns:
        for i, line in enumerate(lines):
            if re.search(pattern, line.strip(), re.IGNORECASE):
                if _is_toc_line(line):
                    continue
                start_line_idx = i
                matched_pattern = pattern
                break
        if start_line_idx is not None:
            break

    if start_line_idx is None:
        return None

    logger.debug(f"[regex] Found {pc_type} at line {start_line_idx} (matched: '{matched_pattern}')")

    collected_lines = []
    char_count = 0
    for line in lines[start_line_idx:]:
        if collected_lines:
            for end_pattern in end_patterns:
                if re.search(end_pattern, line.strip(), re.IGNORECASE) and not _is_toc_line(line):
                    return "\n".join(collected_lines)
        collected_lines.append(line)
        char_count += len(line)
        if char_count >= MAX_SECTION_CHARS:
            logger.warning(f"{pc_type} section truncated at {MAX_SECTION_CHARS:,} chars.")
            break

    return "\n".join(collected_lines)


def find_section_text_llm(full_text: str, pc_type: str) -> str | None:
    """
    LLM-based fallback section finder.
    Extracts candidate header lines, asks the LLM to identify start/end line numbers,
    then returns the text slice between them.
    """
    logger.info(f"[llm-fallback] Locating {pc_type} section via LLM")
    lines = full_text.split("\n")

    # Build a compact list of candidate header lines (numbered/short, non-ToC)
    candidates = []
    for i, line in enumerate(lines):
        stripped = line.strip()
        if (stripped
                and len(stripped) < 150
                and not _is_toc_line(line)
                and (re.match(r'^[\d\.]+\s+\w', stripped) or stripped.isupper())):
            candidates.append(f"{i}: {stripped}")

    if not candidates:
        logger.warning(f"[llm-fallback] No candidate headers found for {pc_type}")
        return None

    # Limit to 300 candidates to stay within context
    candidate_text = "\n".join(candidates[:300])
    description = _PC_TYPE_DESCRIPTIONS.get(pc_type, pc_type)

    prompt = f"""You are analyzing header lines extracted from a clinical trial protocol PDF.

I need to locate the section containing: {description}

Below are candidate section header lines with their line numbers (format "line_number: header text").
Identify the line number where this section STARTS and where it ENDS (i.e. the line number of the 
next top-level section that begins after the target section).

Candidate headers:
{candidate_text}

Return ONLY a JSON object in this exact format (no explanation, no markdown):
{{"start_line": <integer>, "end_line": <integer>}}

If the section cannot be identified, return:
{{"start_line": null, "end_line": null}}"""

    try:
        raw = _invoke_llm(prompt)
        parsed = _parse_json_response(raw, f"{pc_type}-locator")
        start = parsed.get("start_line")
        end = parsed.get("end_line")

        if start is None:
            logger.warning(f"[llm-fallback] LLM could not locate {pc_type} section")
            return None

        logger.info(f"[llm-fallback] {pc_type} located at lines {start}–{end}")

        # Slice the lines and cap at MAX_SECTION_CHARS
        section_lines = lines[start:end] if end else lines[start:]
        section_text = "\n".join(section_lines)
        if len(section_text) > MAX_SECTION_CHARS:
            section_text = section_text[:MAX_SECTION_CHARS]
            logger.warning(f"[llm-fallback] {pc_type} section truncated at {MAX_SECTION_CHARS:,} chars.")
        return section_text

    except Exception as e:
        logger.warning(f"[llm-fallback] LLM section location failed for {pc_type}: {e}")
        return None


def find_section(full_text: str, pc_type: str) -> str | None:
    """
    Try regex-based section finding first; fall back to LLM if regex returns nothing.
    """
    section = find_section_text(full_text, pc_type)
    if section:
        logger.info(f"{pc_type}: section found via regex ({len(section):,} chars)")
        return section

    logger.info(f"{pc_type}: regex found nothing — trying LLM fallback")
    section = find_section_text_llm(full_text, pc_type)
    if section:
        logger.info(f"{pc_type}: section found via LLM ({len(section):,} chars)")
    return section

## LLM Client & JSON Parsing

In [39]:
import unicodedata, os

def _sanitize_text(text: str) -> str:
    """Normalize unicode and drop characters that can't be ASCII-encoded."""
    text = unicodedata.normalize("NFKC", text)
    return text.encode("ascii", errors="ignore").decode("ascii")


def _get_client() -> anthropic.Anthropic:
    key = os.environ.get("ANTHROPIC_API_KEY", "")
    clean_key = key.strip().encode("ascii", errors="ignore").decode("ascii")
    return anthropic.Anthropic(api_key=clean_key)


def _invoke_llm(prompt: str, system_prompt: str = SYSTEM_PROMPT) -> str:
    """Call the LLM and return the raw text response."""
    client = _get_client()
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        system=_sanitize_text(system_prompt),
        messages=[{"role": "user", "content": _sanitize_text(prompt)}],
    )
    return response.content[0].text


def _parse_json_response(raw_response: str, pc_type: str) -> dict:
    text = raw_response.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
        text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ExtractorError(
            f"Failed to parse JSON for {pc_type}. "
            f"JSONDecodeError: {e}. "
            f"Raw LLM response (first 500 chars): {raw_response[:500]}"
        ) from e

## Per-PC-Type Extraction Functions

In [40]:
def extract_pc_obj(section_text: str) -> dict:
    """Extract study objectives and endpoints (PC-OBJ)."""
    logger.info("Extracting PC-OBJ (objectives and endpoints)")
    prompt = PC_OBJ_PROMPT.format(section_text=section_text)
    raw = _invoke_llm(prompt)
    result = _parse_json_response(raw, "PC-OBJ")

    for key in ["primary", "secondary", "safety", "exploratory"]:
        if key not in result:
            raise ExtractorError(
                f"PC-OBJ response missing required key '{key}'. Got keys: {list(result.keys())}"
            )

    total = sum(len(result.get(k, [])) for k in ["primary", "secondary", "safety", "exploratory"])
    logger.info(f"PC-OBJ extracted {total} objectives")
    return result


def extract_pc_elg_crit(section_text: str) -> dict:
    """Extract eligibility criteria (PC-ELG-CRIT)."""
    logger.info("Extracting PC-ELG-CRIT (eligibility criteria)")
    prompt = PC_ELG_CRIT_PROMPT.format(section_text=section_text)
    raw = _invoke_llm(prompt)
    result = _parse_json_response(raw, "PC-ELG-CRIT")

    for key in ["inclusion_criteria", "exclusion_criteria"]:
        if key not in result:
            raise ExtractorError(
                f"PC-ELG-CRIT response missing required key '{key}'. Got keys: {list(result.keys())}"
            )

    n_inc = len(result.get("inclusion_criteria", []))
    n_exc = len(result.get("exclusion_criteria", []))
    logger.info(f"PC-ELG-CRIT extracted {n_inc} inclusion, {n_exc} exclusion criteria")
    return result


def extract_pc_event(section_text: str) -> dict:
    """Extract adverse event information (PC-EVENT)."""
    logger.info("Extracting PC-EVENT (adverse events)")
    prompt = PC_EVENT_PROMPT.format(section_text=section_text)
    raw = _invoke_llm(prompt)
    return _parse_json_response(raw, "PC-EVENT")


def extract_pc_assessment(section_text: str) -> dict:
    """Extract study assessments (PC-ASSESSMENT)."""
    logger.info("Extracting PC-ASSESSMENT (study assessments)")
    prompt = PC_ASSESSMENT_PROMPT.format(section_text=section_text)
    raw = _invoke_llm(prompt)
    return _parse_json_response(raw, "PC-ASSESSMENT")


def extract_pc_exposure(section_text: str) -> dict:
    """Extract study intervention / dosing information (PC-EXPOSURE)."""
    logger.info("Extracting PC-EXPOSURE (study intervention)")
    prompt = PC_EXPOSURE_PROMPT.format(section_text=section_text)
    raw = _invoke_llm(prompt)
    return _parse_json_response(raw, "PC-EXPOSURE")


def extract_pc_conmed(section_text: str) -> dict:
    """Extract concomitant medication information (PC-CONMED)."""
    logger.info("Extracting PC-CONMED (concomitant medications)")
    prompt = PC_CONMED_PROMPT.format(section_text=section_text)
    raw = _invoke_llm(prompt)
    return _parse_json_response(raw, "PC-CONMED")


_EXTRACTORS = {
    "PC-OBJ": extract_pc_obj,
    "PC-ELG-CRIT": extract_pc_elg_crit,
    "PC-EVENT": extract_pc_event,
    "PC-ASSESSMENT": extract_pc_assessment,
    "PC-EXPOSURE": extract_pc_exposure,
    "PC-CONMED": extract_pc_conmed,
}

## Main Pipeline

In [41]:
def extract_protocol_concepts(pdf_path: Path) -> dict:
    """
    Main extraction pipeline. Runs all 6 PC type extractors against the PDF.

    Section finding: regex first, LLM fallback if regex returns nothing.
    Required PC types raise ExtractorError on failure.
    Optional PC types log a warning and are set to {} on failure.
    """
    full_text = extract_text_from_pdf(pdf_path)
    results: dict = {}
    warnings: list[str] = []

    for pc_type in REQUIRED_PC_TYPES + OPTIONAL_PC_TYPES:
        is_required = pc_type in REQUIRED_PC_TYPES
        section_text = find_section(full_text, pc_type)  # regex → LLM fallback

        if section_text is None:
            msg = f"Section not found in PDF for {pc_type} (regex and LLM both failed)"
            if is_required:
                raise ExtractorError(msg)
            else:
                logger.warning(msg)
                warnings.append(msg)
                results[pc_type] = {}
                continue

        extractor = _EXTRACTORS[pc_type]
        try:
            results[pc_type] = extractor(section_text)
        except ExtractorError as e:
            if is_required:
                raise
            else:
                msg = f"Extraction failed for optional {pc_type}: {e}"
                logger.warning(msg)
                warnings.append(msg)
                results[pc_type] = {}

    results["_metadata"] = {
        "pdf_path": str(pdf_path),
        "prompt_version": PROMPT_VERSION,
        "extraction_warnings": warnings,
    }

    if warnings:
        logger.warning(f"Extraction completed with {len(warnings)} warning(s). Check '_metadata.extraction_warnings'.")
    else:
        logger.info("Extraction completed successfully with no warnings.")

    return results


def save_protocol_concepts(concepts: dict, output_path: Path) -> None:
    """Write protocol concepts dict to JSON file."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(concepts, f, indent=2, ensure_ascii=False)
    logger.info(f"Protocol concepts saved to: {output_path}")

## Final Assembly: Study Metadata + Protocol Concepts

In [42]:
def extract_study_metadata(full_text: str) -> dict:
    """
    Extract top-level study metadata from the first ~8000 chars of the PDF
    (title page + synopsis are almost always within the first few pages).
    """
    logger.info("Extracting study-level metadata")
    excerpt = _sanitize_text(full_text[:8000])
    prompt = STUDY_METADATA_PROMPT.format(section_text=excerpt)
    raw = _invoke_llm(prompt)
    result = _parse_json_response(raw, "STUDY_METADATA")
    logger.info(f"Metadata extracted: study_id={result.get('study_id')}, phase={result.get('phase')}")
    return result


def assemble_final_output(concepts: dict, full_text: str) -> dict:
    """
    Combine study metadata with PC type extractions into the expected output format:
    {
      "study_id": ...,
      "acronym": ...,
      ...study metadata fields...,
      "protocol_concepts": [
        {"id": "PC-OBJ",  "type": "OBJECTIVES_AND_ENDPOINTS", "text": {...}},
        {"id": "PC-ELG-CRIT", "type": "ELIGIBILITY_CRITERIA",   "text": {...}},
        ...
      ],
      "_metadata": {...}
    }
    """
    metadata = extract_study_metadata(full_text)

    # Map PC type → display type label
    pc_type_labels = {
        "PC-OBJ":        "OBJECTIVES_AND_ENDPOINTS",
        "PC-ELG-CRIT":   "ELIGIBILITY_CRITERIA",
        "PC-EVENT":      "ADVERSE_EVENTS",
        "PC-ASSESSMENT": "STUDY_ASSESSMENTS",
        "PC-EXPOSURE":   "STUDY_INTERVENTION",
        "PC-CONMED":     "CONCOMITANT_THERAPY",
    }

    protocol_concepts = []
    for pc_type, type_label in pc_type_labels.items():
        if pc_type in concepts:
            protocol_concepts.append({
                "id": pc_type,
                "type": type_label,
                "text": concepts[pc_type],
            })

    return {
        **metadata,
        "protocol_concepts": protocol_concepts,
        "_metadata": concepts.get("_metadata", {}),
    }


def save_final_output(output: dict, output_path: Path) -> None:
    """Write final assembled output to JSON."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    logger.info(f"Final output saved to: {output_path}")

## Run Extraction

Set `PDF_PATH` to your protocol PDF and `OUTPUT_PATH` to where you want the JSON saved.

In [43]:
# ── Edit these two paths ──────────────────────────────────────────────────────
PDF_PATH = Path("/Users/atreyeemukherjee/Downloads/Alexion_NCT04573309_Wilsons.pdf")
OUTPUT_PATH = PDF_PATH.parent / f"{PDF_PATH.stem}_protocol_concepts.json"
# ─────────────────────────────────────────────────────────────────────────────
print(f"PDF:    {PDF_PATH}")
print(f"Output: {OUTPUT_PATH}")

PDF:    /Users/atreyeemukherjee/Downloads/Alexion_NCT04573309_Wilsons.pdf
Output: /Users/atreyeemukherjee/Downloads/Alexion_NCT04573309_Wilsons_protocol_concepts.json


In [47]:
# Step 1: extract all PC types
full_text = extract_text_from_pdf(PDF_PATH)
concepts = extract_protocol_concepts(PDF_PATH)

# Step 2: assemble with study metadata into final format
final_output = assemble_final_output(concepts, full_text)
save_final_output(final_output, OUTPUT_PATH)

print(f"\nStudy: {final_output.get('study_id')} — {final_output.get('phase')}")
print(f"Sponsor: {final_output.get('sponsor_name')}")
print(f"PC types extracted: {[pc['id'] for pc in final_output['protocol_concepts']]}")
if final_output["_metadata"]["extraction_warnings"]:
    print("\nWarnings:")
    for w in final_output["_metadata"]["extraction_warnings"]:
        print(f"  - {w}")

2026-03-09 20:48:08,234 [INFO] __main__: Extracting text from PDF: /Users/atreyeemukherjee/Downloads/Alexion_NCT04573309_Wilsons.pdf
2026-03-09 20:48:08,237 [INFO] __main__: PDF has 74 pages
2026-03-09 20:48:08,385 [INFO] __main__: Extracted 187,555 characters from PDF
2026-03-09 20:48:08,385 [INFO] __main__: Extracting text from PDF: /Users/atreyeemukherjee/Downloads/Alexion_NCT04573309_Wilsons.pdf
2026-03-09 20:48:08,388 [INFO] __main__: PDF has 74 pages
2026-03-09 20:48:08,535 [INFO] __main__: Extracted 187,555 characters from PDF
2026-03-09 20:48:08,539 [WARNING] __main__: PC-OBJ section truncated at 12,000 chars.
2026-03-09 20:48:08,540 [INFO] __main__: PC-OBJ: section found via regex (13,057 chars)
2026-03-09 20:48:08,540 [INFO] __main__: Extracting PC-OBJ (objectives and endpoints)
2026-03-09 20:48:28,575 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-03-09 20:48:28,579 [INFO] __main__: PC-OBJ extracted 14 objectives
2026-03-09 20:4


Study: ALXN1840-WD-204 — Phase 2
Sponsor: Alexion Pharmaceuticals, Inc.
PC types extracted: ['PC-OBJ', 'PC-ELG-CRIT', 'PC-EVENT', 'PC-ASSESSMENT', 'PC-EXPOSURE', 'PC-CONMED']
